In [16]:
import warnings 
warnings.filterwarnings('ignore')

# Document load 
from langchain_community.document_loaders import PyPDFLoader 
loader  = PyPDFLoader('Static GK 2025.pdf')
pages = loader.load()

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
import hashlib

# Split Data 

spliter = RecursiveCharacterTextSplitter(chunk_size=1400 , chunk_overlap=180)
text_spliter = spliter.split_documents(pages)
chunks = [i.page_content for i in text_spliter]
metadata = [i.metadata for i in text_spliter]
ids = [hashlib.md5(chunk.encode('utf-8')).hexdigest() for chunk in chunks]
print(f'print first 5 ids : {ids[:2]}')

print first 5 ids : ['df52eef7bfa55759b4642211e13e3020', '622d6c3b19974d6f39f9950848df1607']


In [36]:
import chromadb 
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 
embedding_function = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# client and collection create 
client = chromadb.PersistentClient(path="./Agentic_RAG_Database")
collection = client.get_or_create_collection(name="Agentic_Rag",embedding_function=embedding_function)

if chunks:
    collection.add(
        ids=ids,
        documents=chunks , metadatas=metadata
    )
collection.count()

225

In [37]:
# LLM call 
import os 
from dotenv import load_dotenv 
from langchain_groq import ChatGroq 
load_dotenv()
try:
    key = os.getenv('GROQ_API_KEY')
    print(bool(key))
except Exception as e:
    print(str(e))
    
Groq = ChatGroq(model="qwen/qwen3.6-27b")

test = Groq.invoke("hello llama?")
test.content

True


'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user said "hello llama?"\n   - This is a greeting ("hello") followed by "llama" with a question mark, likely referencing the AI model name "Llama" or just a playful/casual greeting.\n   - I need to clarify my identity: I am Qwen, developed by Alibaba Group\'s Tongyi Lab, not Llama.\n\n2.  **Identify Key Requirements:**\n   - Acknowledge the greeting politely.\n   - Clarify my identity accurately (I\'m Qwen, not Llama).\n   - Maintain a friendly, helpful tone.\n   - Keep it concise.\n\n3.  **Formulate Response (Mental Refinement):**\n   - "Hello! I\'m actually Qwen, not Llama. How can I assist you today?"\n   - Check against guidelines: Accurate identity? Yes. Clear and concise? Yes. Friendly? Yes. Matches language? Yes (English). No extra fluff.\n\n4.  **Final Output Generation:** (Matches the refined version)\n   "Hello! I\'m actually Qwen, not Llama. How can I help you today?"✅\n   - Self-Correction/Verificati

In [38]:
# Hybrid Corpus 
from rank_bm25 import BM25Okapi 
def tokenization(token):
    token = token.lower()
    token = token.split()
    return token 

tokens = [tokenization(i) for i in chunks]
bm_corpus = BM25Okapi(tokens)

print(f'sucussfully : {bm_corpus}')

sucussfully : <rank_bm25.BM25Okapi object at 0x12e280e90>


In [39]:
# Tool creation 

# Calculator 
import numexpr 
from  langchain_community.tools import tool 
@tool 
def calculator(execution:str):
    """ User given anytypes of arithmetic calculation done by this tool """
    try:
        response = numexpr.evaluate(execution).item()
        return response 
    except Exception as e :
        return str(e)
    
# retrival 
@tool
def Hybrid_Retrive(query:str):
    """THIS IS local given document by user . so , user asking all sort of questions is passing through this tool"""
    query_re = Groq.invoke(f"write the query based on symentic search : {query}").content.strip()

    # Thats Vector DB retrival 
    result = collection.query(query_texts=[query_re] , n_results=5)
    dis  = result['distances'][0] 
    docs = result['documents'][0]
    threshold = 0.9
    print(f'the distance is : {dis}')
    dense_docs = []
    for i , d in zip(dis,docs):
        if threshold > i :
            dense_docs.append(d)
    # Thats Hybrid RAG Retrival using indexing 
    query_tokens = tokenization(query_re)
    score = bm_corpus.get_scores(query=query_tokens)
    def get_top_tokens (score , k=10):
        index = list(enumerate(score))
        idx_sorted = sorted(index,key=lambda x:x[1],reverse=True)
        return [doc for doc , _ in idx_sorted[:10]]
    index_tokens = [chunks[i] for i in get_top_tokens(score=score,k=10)]
    
    rrf_token = {}
    
    for rank , doc in enumerate(dense_docs):
        rrf_token[doc] = rrf_token.get(doc,0)+1/(rank+60)
    for rank , doc in enumerate(index_tokens):
        rrf_token[doc] = rrf_token.get(doc,0)+1/(rank+60)
        
    marge = sorted(rrf_token.items() , key = lambda x:x[1] , reverse=True)
    get_docs = [i for i , _ in marge[:5]]
    
    return "\n\n".join(get_docs)


In [40]:
tools = [calculator,Hybrid_Retrive]
tool_name  = {t.name : t for t in tools}
print(tool_name)

{'calculator': StructuredTool(name='calculator', description='User given anytypes of arithmetic calculation done by this tool', args_schema=<class 'langchain_core.utils.pydantic.calculator'>, func=<function calculator at 0x12de7f560>), 'Hybrid_Retrive': StructuredTool(name='Hybrid_Retrive', description='THIS IS local given document by user . so , user asking all sort of questions is passing through this tool', args_schema=<class 'langchain_core.utils.pydantic.Hybrid_Retrive'>, func=<function Hybrid_Retrive at 0x12de7ee80>)}


In [41]:
# Tool conection with LLM 
try:
    llm_tool_blind = Groq.bind_tools(tools=tools)
    print(bool(llm_tool_blind))
except Exception as e :
    print(str(e))

True


In [42]:
def search_tool(question: str):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]
    
    # 1. Initialize context list at the top (matching name used later)
    retrieved_contexts = []
    
    response = llm_tool_blind.invoke(messages)
    
    # Direct answer path (No tool calls)
    if not response.tool_calls:
        ans = response.content if response.content else "no response generate"
        return ans, ["N/A - direct answer"]
    
    messages.append(response)
    
    # Tool execution path
    for call in response.tool_calls:
        name = call['name']
        arguments = call['args']
        
        tool_response = tool_name[name].invoke(arguments)
        
        # Capture context chunks from Hybrid_Retrive
        if name == "Hybrid_Retrive":
            chunks = [c.strip() for c in str(tool_response).split("\n\n") if c.strip()]
            retrieved_contexts.extend(chunks)
        
        messages.append({
            "tool_call_id": call['id'],
            "content": str(tool_response),
            "role": "tool"
        })
    
    # 2. Fallback check for non-retrieval tools (e.g., calculator)
    if not retrieved_contexts:
        retrieved_contexts = ["N/A - Math Calculation"]
        
    result = Groq.invoke(messages)
    ans = result.content if result.content else "no response generate"
    
    return ans, retrieved_contexts

In [43]:
from datasets import Dataset 
from ragas import evaluate 
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from langchain_community.embeddings import HuggingFaceEmbeddings


test_cases = [
    {
        "question": "who is first First Chief of Army Staff",
        "ground_truth": "General Maharaj Rajendra Singh Ji was the first Chief of Army Staff."
    },
    {
        "question": "Calculate 45 * 12 + 150",
        "ground_truth": "690"
    }
]

user_input = []
retrive_context = []
response =  []
reference = [] 

for item in test_cases:
    qus = item['question']
    truth = item['ground_truth']
    
    answer , context = search_tool(question=qus)
    print(answer)
    
    user_input.append(qus)
    retrive_context.append(context)
    response.append(answer)
    reference.append(truth)
    
data = {
    "user_input":user_input , 
    "retrieved_contexts":retrive_context ,
    "response":response , 
    "reference":reference
}

embedding = HuggingFaceEmbeddings(model_name= "all-MiniLM-L6-v2")
dataset = Dataset.from_dict(data)

result = evaluate(
    dataset = dataset , 
    metrics=[Faithfulness(), AnswerRelevancy(), ContextPrecision(), ContextRecall()] , 
    embeddings=embedding , 
    llm=Groq
)

print(result)


the distance is : [0.648424506187439, 0.6916962265968323, 0.6968715786933899, 0.7007758617401123, 0.7042617797851562]

<think>
The user wants to know who the first Chief of Army Staff was.
The search results contain a snippet from a document titled "GK Now – Current Affairs".
In the section "First in World Male", item number 4 says: "First Chief of Army Staff General Maharaj Rajendra Singh Ji".
This likely refers to the Indian Army, given the context of other entries like "First Field Marshal of India SHFJ Manekshaw", "First Governor-General of India William Bentinck", etc.
So, the first Chief of Army Staff of the Indian Army was General Maharaj Rajendra Singh Ji.
I should confirm this fact. General Maharaj Rajendra Singh of Jaipur was the first Indian Chief of Army Staff.
Wait, let's double check.
"General Maharaj Rajendra Singh Ji" is indeed General Maharaj Rajendra Singh of Jaipur, who served as the Chief of the Army Staff of the Indian Army from 1949 to 1954. He was the first India

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/debajyotihazra/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x108ddf740> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/debajyotihazra/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x108ddf740> is already entered
Task was destroyed but it is pending!
task: <Task pending name='Task-503' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/debajyotihazra/Documents/MultiAgentic RAG /.venv/lib/python3.12/si

{'faithfulness': nan, 'answer_relevancy': nan, 'context_precision': 1.0000, 'context_recall': 1.0000}
